# 🦾 Titan Browser — GitHub Actions Runner on Colab

Ce notebook transforme ta session Colab en **self-hosted runner** pour GitHub Actions.  
Le build Chromium tourne ici (jusqu'à 12h session gratuite) au lieu du runner GitHub (limité à 6h).

### Étapes
1. **Cellule 1** — Entre ton `GITHUB_PAT` quand demandé (saisie sécurisée)
2. **Cellule 2** — Vérifie les ressources disponibles
3. **Cellule 3** — Installe le runner
4. **Cellule 4** — Enregistre et démarre le runner
5. Dans GitHub → repo → Actions → **Run workflow** → sélectionne **`self-hosted`**

> ⚠️ **Ne ferme pas l'onglet Colab** pendant le build.
> 💡 **Astuce** : Sauvegarde ton PAT dans **Tools → Secrets → `GITHUB_PAT`** pour ne plus avoir à le retaper.

In [ ]:
# ── Cellule 1 : Configuration ─────────────────────────────────────────────────
import os, getpass

REPO          = "ferelking242/titan-browser"
RUNNER_NAME   = "colab-runner"
RUNNER_LABELS = "self-hosted,colab,linux,x64"

# Méthode 1 : Secrets Colab (Tools → Secrets → ajoute GITHUB_PAT)
GITHUB_PAT = ""
try:
    from google.colab import userdata
    GITHUB_PAT = userdata.get('GITHUB_PAT') or ""
    if GITHUB_PAT:
        print("✅ PAT chargé depuis les Secrets Colab")
except Exception:
    pass

# Méthode 2 : saisie sécurisée interactive
if not GITHUB_PAT:
    GITHUB_PAT = getpass.getpass("🔑 Colle ton GitHub PAT (scopes: repo + workflow) : ")

if not GITHUB_PAT:
    raise ValueError("❌ PAT vide — relance la cellule et entre ton token")

os.environ['GITHUB_PAT'] = GITHUB_PAT
print(f"✅ Config OK — Repo : {REPO} | Runner : {RUNNER_NAME}")

In [ ]:
# ── Cellule 2 : Vérification des ressources ───────────────────────────────────
import subprocess, shutil

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

print("=== CPU ===")
print(sh("nproc && lscpu | grep 'Model name' | head -1"))
print("\n=== RAM ===")
print(sh("free -h | head -2"))
print("\n=== Disk ===")
print(sh("df -h / | tail -1"))

_, _, free = shutil.disk_usage("/")
free_gb = free / (1024**3)
if free_gb < 60:
    print(f"\n⚠️  Seulement {free_gb:.0f}GB libres — Chromium en a besoin de ~80GB.")
    print("   Dans Colab : Runtime → Disconnect and delete runtime, puis reconnecte.")
else:
    print(f"\n✅ {free_gb:.0f}GB libres — suffisant pour le build")

In [ ]:
# ── Cellule 3 : Installation du runner GitHub Actions ─────────────────────────
import requests, subprocess, os, urllib.request

RUNNER_VERSION = "2.325.0"
RUNNER_DIR     = "/root/actions-runner"

print("Récupération du token d'enregistrement...")
resp = requests.post(
    f"https://api.github.com/repos/{REPO}/actions/runners/registration-token",
    headers={"Authorization": f"token {GITHUB_PAT}", "Accept": "application/vnd.github.v3+json"}
)
assert resp.status_code == 201, f"❌ Erreur API: {resp.status_code} — {resp.text}"
reg_token = resp.json()["token"]
print(f"✅ Token obtenu (expire: {resp.json()['expires_at']})")

os.makedirs(RUNNER_DIR, exist_ok=True)
runner_url = f"https://github.com/actions/runner/releases/download/v{RUNNER_VERSION}/actions-runner-linux-x64-{RUNNER_VERSION}.tar.gz"
runner_tar = "/tmp/actions-runner.tar.gz"

print(f"\nTéléchargement du runner v{RUNNER_VERSION}...")
urllib.request.urlretrieve(runner_url, runner_tar)
print("✅ Téléchargé")

print("Extraction...")
subprocess.run(["tar", "xzf", runner_tar, "-C", RUNNER_DIR], check=True)
print("✅ Extrait")

print("\nInstallation des dépendances runner...")
subprocess.run(f"cd {RUNNER_DIR} && bash bin/installdependencies.sh", shell=True, check=True, capture_output=True)
print("✅ Dépendances OK")

In [ ]:
# ── Cellule 4 : Enregistrement + démarrage du runner ─────────────────────────
import subprocess

RUNNER_DIR = "/root/actions-runner"

print("Enregistrement du runner...")
reg_cmd = (
    f"cd {RUNNER_DIR} && ./config.sh "
    f"--url https://github.com/{REPO} "
    f"--token {reg_token} "
    f"--name {RUNNER_NAME} "
    f"--labels '{RUNNER_LABELS}' "
    f"--work /root/_work "
    f"--replace --unattended"
)
result = subprocess.run(reg_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ Erreur:", result.stderr)
    raise RuntimeError("Enregistrement échoué")
print("✅ Runner enregistré")

print("\nDémarrage du runner (Ctrl+C pour arrêter)...")
print("─" * 60)
print("👉 Va sur GitHub → ton repo → Actions → Run workflow")
print("   Sélectionne runner: self-hosted (ou colab)")
print("─" * 60)

proc = subprocess.Popen(
    f"cd {RUNNER_DIR} && ./run.sh",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    for line in proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    proc.terminate()
    print("\n🛑 Runner arrêté")

---
## ℹ️ Notes importantes

### Ressources Colab
| Tier | CPU | RAM | Session max |
|------|-----|-----|-------------|
| Gratuit | 2 vCPU | 12 GB | ~12h |
| Pro | 2-4 vCPU | 25 GB | 24h |
| Pro+ | 4-8 vCPU | 52 GB | 24h |

Avec ccache chaud (2ème build), n'importe quel tier suffit (~1-2h).

### Déclencher le build
```
GitHub → ferelking242/titan-browser → Actions → Build → Run workflow
Runner: self-hosted
Full build: ☐ (arm64 only)
```